In [0]:
# Bronze — Dallas Raw Ingestion
# Purpose : Read Dallas CSV → rename columns → append to bronze.dallas_raw
#  Bronze preserves raw data exactly as-is.
# inferSchema=false intentional — Bronze preserves raw data exactly as-is

In [0]:
import uuid
from datetime import datetime
from pyspark.sql import functions as F

# Detect catalog dynamically — no hardcoded catalog name
CATALOG = spark.sql("SELECT current_catalog()").first()[0]
if CATALOG in ("hive_metastore", "spark_catalog"):
    catalogs = [r[0] for r in spark.sql("SHOW CATALOGS").collect()]
    project_cats = [c for c in catalogs
                    if c not in ("hive_metastore", "spark_catalog",
                                 "system", "samples", "workspace")]
    CATALOG = project_cats[0] if project_cats else CATALOG

# Paths — built from catalog, no hardcoding
DAL_PATH     = f"/Volumes/{CATALOG}/bronze/raw_data/Dallas_Inspections_20260408.csv"
BRONZE_TABLE = f"{CATALOG}.bronze.dallas_raw"

# Batch metadata — unique per run
# Multiple runs same day = different BATCH_ID and _extract_ts
# Silver always reads latest _batch_id — history never lost
BATCH_ID         = str(uuid.uuid4())
EXTRACT_TS_STR   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
EXTRACT_DATE_STR = datetime.now().strftime("%Y-%m-%d")

print(f"Catalog      : {CATALOG}")
print(f"Batch ID     : {BATCH_ID}")
print(f"Extract time : {EXTRACT_TS_STR}")
print(f"Extract date : {EXTRACT_DATE_STR}")
print(f"Target table : {BRONZE_TABLE}")

In [0]:
# Confirm file is in Volume before reading
files = [f.name for f in dbutils.fs.ls(f"/Volumes/{CATALOG}/bronze/raw_data/")]
print("Files in Volume:")
for f in files:
    print(f"  {f}")

assert "Dallas_Inspections_20260408.csv" in files, \
    "ERROR: Dallas CSV not found — check Volume path or filename!"
print("\n✓ Dallas file confirmed — safe to proceed")

In [0]:
# inferSchema=false is deliberate — Bronze preserves raw data exactly as-is
# Type casting happens in Silver, never here
df_dal_raw = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "false")
    .option("multiLine",   "true")
    .option("escape",      '"')
    .option("encoding",    "UTF-8")
    .csv(DAL_PATH)
)

print(f"Row count : {df_dal_raw.count():,}")
print(f"Col count : {len(df_dal_raw.columns)}")
# Expected: 78,984 rows | 114 columns

In [0]:
# Dallas source has a known typo in column 20
# 'Violation  Memo - 20' has TWO spaces — must fix before the rename loop
bad_col  = "Violation  Memo - 20"
good_col = "Violation Memo - 20"

if bad_col in df_dal_raw.columns:
    df_dal_raw = df_dal_raw.withColumnRenamed(bad_col, good_col)
    print(f"✓ Fixed typo: '{bad_col}' → '{good_col}'")
else:
    print("Column already clean — no fix needed")

print(f"Total columns: {len(df_dal_raw.columns)}")

In [0]:
# Step 1: Rename core columns to snake_case
df_dal = (
    df_dal_raw
    .withColumnRenamed("Restaurant Name",   "restaurant_name")
    .withColumnRenamed("Inspection Type",   "inspection_type")
    .withColumnRenamed("Inspection Date",   "inspection_date")
    .withColumnRenamed("Inspection Score",  "inspection_score")
    .withColumnRenamed("Street Number",     "street_number")
    .withColumnRenamed("Street Name",       "street_name")
    .withColumnRenamed("Street Direction",  "street_direction")
    .withColumnRenamed("Street Type",       "street_type")
    .withColumnRenamed("Street Unit",       "street_unit")
    .withColumnRenamed("Street Address",    "street_address")
    .withColumnRenamed("Zip Code",          "zip_code")
    .withColumnRenamed("Inspection Month",  "inspection_month")
    .withColumnRenamed("Inspection Year",   "inspection_year")
    .withColumnRenamed("Lat Long Location", "location_raw")
)

# Step 2: Rename all 25 violation sets to Delta-safe names
# Original names have spaces and dashes — Delta rejects them
# Silver will unpivot these into long format
for i in range(1, 26):
    df_dal = (
        df_dal
        .withColumnRenamed(f"Violation Description - {i}", f"viol_desc_{i}")
        .withColumnRenamed(f"Violation Points - {i}",      f"viol_points_{i}")
        .withColumnRenamed(f"Violation Detail - {i}",      f"viol_detail_{i}")
        .withColumnRenamed(f"Violation Memo - {i}",        f"viol_memo_{i}")
    )

# Step 3: Add audit metadata only — no null placeholders for Chicago columns
# Dallas bronze stays Dallas-only. Silver handles the merge logic.
df_dal = (
    df_dal
    .withColumn("city_source",   F.lit("DAL"))
    .withColumn("_batch_id",     F.lit(BATCH_ID))
    .withColumn("_extract_ts",   F.lit(EXTRACT_TS_STR))
    .withColumn("_extract_date", F.lit(EXTRACT_DATE_STR))
    .withColumn("_source_file",  F.lit("Dallas_Inspections_20260408.csv"))
)

# Verify no special characters remain in column names
bad_cols = [c for c in df_dal.columns
            if any(x in c for x in [" ", "-", "(", ")", ","])]
if bad_cols:
    print(f"WARNING — still bad columns: {bad_cols}")
else:
    print(f"✓ All column names are Delta-safe")
    print(f"  Total columns : {len(df_dal.columns)}")

display(df_dal.select(
    "restaurant_name", "inspection_date", "inspection_score",
    "viol_desc_1", "viol_points_1", "city_source", "_batch_id"
).limit(3))

In [0]:
# Write — APPEND mode
# History preserved via _extract_date + _batch_id + _extract_ts columns
# Multiple runs on same day = different _batch_id and _extract_ts rows
# APPEND — never overwrites, every run adds a new batch
# History identified by _batch_id + _extract_ts
(
    df_dal
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

# Verify this exact batch landed
written = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == BATCH_ID)
    .count()
)
source = df_dal.count()

print(f"✓ Written to  : {BRONZE_TABLE}")
print(f"  Batch ID    : {BATCH_ID}")
print(f"  Extract ts  : {EXTRACT_TS_STR}")
print(f"  Source rows : {source:,}")
print(f"  Written rows: {written:,}")
print(f"  Match       : {'✓' if source == written else 'MISMATCH'}")

In [0]:
# Show every batch ever loaded — proof of history preservation
df_batches = spark.sql(f"""
    SELECT
        city_source,
        _extract_date,
        _extract_ts,
        LEFT(_batch_id, 8)  AS batch_short,
        COUNT(*)            AS row_count
    FROM {BRONZE_TABLE}
    GROUP BY city_source, _extract_date, _extract_ts, _batch_id
    ORDER BY _extract_ts DESC
""")
display(df_batches)

# Delta version history — every append = new version
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))